In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")

from dotenv import load_dotenv
load_dotenv()
DB_URL = os.getenv("DB_URL")

import numpy as np
import pandas as pd
import wrds
from sqlalchemy import create_engine, text
import seaborn as sns
import matplotlib.pyplot as plt
import polars as pl

START_DATE = "1990-01-01"
END_DATE   = "2025-12-31"

PCT_LO = 5
PCT_HI = 20


In [38]:
db = wrds.Connection()

WRDS recommends setting up a .pgpass file.
pgpass file created at C:\Users\reyno\AppData\Roaming\postgresql\pgpass.conf
Created .pgpass file successfully.
You can create this file yourself at any time with the create_pgpass_file() function.
Loading library list...
Done


In [24]:
crsp = db.raw_sql("""
    SELECT
        a.permno,
        a.date,
        a.ret,
        a.prc,
        a.shrout,
        ABS(a.prc) * a.shrout   AS mktcap,
        a.vol,
        b.exchcd,
        b.shrcd,
        b.cusip,
        b.ticker,
        b.comnam
    FROM crsp.msf      AS a
    JOIN crsp.msenames AS b
      ON  a.permno  = b.permno
      AND b.namedt <= a.date
      AND a.date   <= COALESCE(b.nameendt, CURRENT_DATE)
    WHERE a.date   BETWEEN '1990-01-01' AND '2025-12-31'
      AND b.shrcd  IN (10, 11)
      AND b.exchcd IN (1, 2, 3)  
""", date_cols=["date"])

crsp["mktcap"] = crsp["mktcap"].abs() / 1000  #convert to millions

#delisting returns
dlret = db.raw_sql("""
    SELECT permno, dlstdt as date, dlret, dlstcd
    FROM crsp.msedelist
    WHERE dlstdt BETWEEN '1990-01-01' AND '2025-12-31'
""", date_cols=["date"])

# Create month columns for merging
crsp["month"] = pd.to_datetime(crsp["date"]).dt.to_period('M').dt.to_timestamp()
dlret["month"] = pd.to_datetime(dlret["date"]).dt.to_period('M').dt.to_timestamp()

crsp = crsp.merge(dlret[['permno', 'month', 'dlret', 'dlstcd']], on=['permno', 'month'], how='left')

# Shumway (1997) adjustment: -30% for performance delists (500, 520-584) if missing
perf_delist = crsp['dlstcd'].between(500, 584)
crsp.loc[perf_delist & crsp['dlret'].isna(), 'dlret'] = -0.30

crsp['ret'] = crsp['ret'].fillna(0)
crsp['dlret'] = crsp['dlret'].fillna(0)
crsp['ret_adj'] = (1 + crsp['ret']) * (1 + crsp['dlret']) - 1

crsp['ret'] = crsp['ret_adj']
crsp = crsp.drop(columns=['dlret', 'dlstcd', 'ret_adj'])

print(crsp.shape)
crsp.head()

(2082485, 13)


,permno,date,ret,prc,shrout,mktcap,vol,exchcd,shrcd,cusip,ticker,comnam,month
0,10066,1999-03-31,0.0,2.9375,24699.0,72.553313,18463.0,2,11,35518410,FCM,FRANKLIN TELECOMMUNICATIONS CORP,1999-03-01
1,10066,1999-04-30,0.0,2.9375,24699.0,72.553313,23648.0,2,11,35518410,FCM,FRANKLIN TELECOMMUNICATIONS CORP,1999-04-01
2,10066,1999-05-28,-0.148936,2.5,25423.0,63.5575,15695.0,2,11,35518410,FCM,FRANKLIN TELECOMMUNICATIONS CORP,1999-05-01
3,10066,1999-06-30,0.05,2.625,25207.0,66.168375,16582.0,2,11,35518410,FCM,FRANKLIN TELECOMMUNICATIONS CORP,1999-06-01
4,10066,1999-07-30,-0.190476,2.125,25207.0,53.564875,13986.0,2,11,35518410,FCM,FRANKLIN TELECOMMUNICATIONS CORP,1999-07-01


In [ ]:
#Delisting Diagnostic
penalty_hits = crsp[np.isclose(crsp['ret'], -0.30, atol=1e-9)]
shumway_hits = penalty_hits.shape[0]

print(f"--- Delisting Diagnostic ---")
print(f"Total Shumway -30% Penalties Applied: {shumway_hits}")

if shumway_hits > 0:
    print("\nIntegrity Check: First 5 Shumway Cases")
    print(penalty_hits[['permno', 'month', 'ret']].head())
else:
    print("Zero Shumway hits. CHECK PIPELINE TS NOT RIGHT.")
    print("Check floating point drift")

NameError: name 'crsp' is not defined

In [26]:
rf = db.raw_sql("""
    SELECT date, rf
    FROM ff.factors_monthly
    WHERE date BETWEEN '1990-01-01' AND '2025-12-31'
""", date_cols=["date"])

print(rf.shape)
rf.head()

(432, 2)


,date,rf
0,1990-01-01,0.0057
1,1990-02-01,0.0057
2,1990-03-01,0.0064
3,1990-04-01,0.0069
4,1990-05-01,0.0068


In [27]:
comp = db.raw_sql("""
    SELECT
        gvkey,
        cusip,
        datadate,
        at      AS total_assets,
        ceq     AS book_equity,
        ni      AS net_income,
        gp      AS gross_profit,
        dltt    AS long_term_debt,
        dlc     AS short_term_debt
    FROM comp.funda
    WHERE datadate BETWEEN '1990-01-01' AND '2025-12-31'
      AND indfmt  = 'INDL'
      AND datafmt = 'STD'
      AND popsrc  = 'D'
      AND consol  = 'C'
      AND at > 0
""", date_cols=["datadate"])

print(comp.shape)
comp.head()

(328182, 9)


,gvkey,cusip,datadate,total_assets,book_equity,net_income,gross_profit,long_term_debt,short_term_debt
0,001003,000354100,1990-01-31,10.109,-0.416,-0.221,7.384,0.076,4.449
1,001004,000361105,1990-05-31,388.521,189.548,25.655,107.944,72.329,33.821
2,001004,000361105,1991-05-31,379.958,193.778,14.801,100.502,68.953,16.5
3,001004,000361105,1992-05-31,395.351,196.737,10.02,91.601,67.323,25.005
4,001004,000361105,1993-05-31,365.151,189.216,0.283,75.344,66.298,25.025


In [28]:
bidask = db.raw_sql("""
    SELECT
        permno,
        DATE_TRUNC('month', date)::date  AS date,
        AVG(
            CASE WHEN askhi + bidlo > 0
                 THEN (askhi - bidlo) / ((askhi + bidlo) / 2.0)
                 ELSE NULL END
        ) AS bidask
    FROM crsp.dsf
    WHERE date BETWEEN '1990-01-01' AND '2025-12-31'
      AND askhi IS NOT NULL
      AND bidlo IS NOT NULL
      AND askhi >= bidlo
    GROUP BY permno, DATE_TRUNC('month', date)
""", date_cols=["date"])

print(bidask.shape)
bidask.head()

(3199332, 3)


,permno,date,bidask
0,26650,1994-06-01,0.00982
1,85885,1992-06-01,0.035687
2,14526,1993-05-01,0.038776
3,85164,1994-04-01,0.185589
4,37875,1990-04-01,0.028655


In [29]:
db.close()

In [30]:
for d in [crsp, rf, comp, bidask]:
    date_col = 'date' if 'date' in d.columns else 'datadate'
    d["month"] = pd.to_datetime(d[date_col]).dt.to_period('M').dt.to_timestamp()

In [31]:
df = crsp.merge(rf[["month", "rf"]], on="month", how="left")
df["excess_ret"] = df["ret"] - df["rf"].fillna(0)
df = df.merge(bidask[["permno", "month", "bidask"]], on=["permno", "month"], how="left")

In [32]:
comp["merge_month"] = (comp["month"] + pd.DateOffset(months=6))
comp["cusip8"] = comp["cusip"].str[:8]
df["cusip8"]   = df["cusip"].str[:8]

comp_cols = ["cusip8", "merge_month", "total_assets", "book_equity", 
             "net_income", "gross_profit", "long_term_debt", "short_term_debt"]

df = df.merge(
    comp[comp_cols],
    left_on=["cusip8", "month"], 
    right_on=["cusip8", "merge_month"], 
    how="left"
).drop(columns=["merge_month"])

In [33]:
#fill annualized for high missigness characteristics
df = df.sort_values(["permno", "month"])

fund_vars = ["total_assets", "book_equity", "net_income", "gross_profit", "long_term_debt", "short_term_debt"]
df[fund_vars] = df.groupby("permno")[fund_vars].ffill(limit=12)

In [34]:
group = df.groupby("permno")

# Size & Momentum
df["mktcap_lag"] = group["mktcap"].shift(1)
df["size"] = np.log(df["mktcap_lag"])
df["momentum"] = group["ret"].transform(lambda x: x.shift(2).rolling(11).apply(lambda r: (1+r).prod()-1, raw=True))

# Fundamentals
df["bm"] = df["book_equity"] / df["mktcap_lag"]
df["roa"] = df["net_income"] / df["total_assets"]
df["leverage"] = (df["long_term_debt"].fillna(0) + df["short_term_debt"].fillna(0)) / df["mktcap_lag"]

In [4]:
df = pl.read_parquet("data/microcap.parquet").to_pandas()

In [5]:
CHARACTERISTICS = ["ret", "prc", "shrout", "mktcap", "vol", "bidask", "excess_ret", "total_assets", "book_equity", "net_income", "gross_profit", "long_term_debt", "short_term_debt", "mktcap_lag", "size", "momentum", "bm", "roa", "leverage"] 

def winsorize(s, lo=0.01, hi=0.99):
    low, high = s.quantile(lo), s.quantile(hi)
    return s.clip(low, high)

def rank_std(s):
    n = s.notna().sum()
    if n < 10: return pd.Series(np.nan, index=s.index)
    return s.rank(method="average", na_option="keep") / (n + 1) - 0.5

df[CHARACTERISTICS] = df.groupby("month")[CHARACTERISTICS].transform(winsorize)
df[CHARACTERISTICS] = df.groupby("month")[CHARACTERISTICS].transform(rank_std)

In [6]:
pl.from_pandas(df.dropna(subset=["excess_ret"])).write_parquet("microcapStandardized.parquet")

print("Pipeline executed successfuly -> microcapStandardized.parquet")

Pipeline executed successfuly -> microcapStandardized.parquet


In [35]:
nyse_breakpoints = (
    df[df['exchcd'] == 1]
    .groupby('month')['mktcap_lag']
    .quantile([PCT_LO / 100, PCT_HI / 100])
    .unstack()
)

nyse_breakpoints.columns = ['size_lower_bound', 'size_upper_bound']
df = df.merge(nyse_breakpoints, on='month', how='left')


df = df[
    (df['mktcap_lag'] >= df['size_lower_bound']) & 
    (df['mktcap_lag'] <= df['size_upper_bound'])
]

df = df.drop(columns=['size_lower_bound', 'size_upper_bound']).dropna(subset=["excess_ret"])

print(f"Universe filtered! Remaining observations: {len(df)}")

Universe filtered! Remaining observations: 505086


In [36]:
pl.from_pandas(df.dropna(subset=["excess_ret"])).write_parquet("microcap.parquet")

print("Pipeline executed successfuly -> microcap.parquet")

Pipeline executed successfuly -> microcap.parquet


In [39]:
ff = db.get_table(library='ff', table='factors_monthly')

ff['month'] = pd.to_datetime(ff['date']).dt.to_period('M').dt.to_timestamp()

for col in ['mktrf', 'smb', 'hml', 'rf']:
    ff[col] = ff[col] / 100
    
pl.from_pandas(ff[['month', 'mktrf', 'smb', 'hml', 'rf']]).write_parquet("ff_factors.parquet")

db.close()
print("FF factors saved to ff_factors.parquet")

FF factors saved to ff_factors.parquet
